In [1]:
import ipywidgets as widgets

import QuantLib as ql
import curve_db as cdb
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates

from IPython.display import HTML

HTML('''<script>
code_show=true; 
function code_toggle() {
 if (code_show){
 $('div.input').hide();
 } else {
 $('div.input').show();
 }
 code_show = !code_show
} 
$( document ).ready(code_toggle);
</script>
<form action="javascript:code_toggle()"><input type="submit" value="Click here to toggle on/off the raw code."></form>''')

In [2]:
cal_b3 = ql.Brazil(ql.Brazil().Exchange)
today = cal_b3.adjust(ql.Date.todaysDate(), ql.Preceding)


dt_p_base = widgets.DatePicker(
    description='Base Curve: ',
    disabled=False,
    value = today.to_date(),
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='300px', height='30px')
)

dt_p = widgets.DatePicker(
    description='Historical Curve Date: ',
    disabled=False,
    value = datetime.date(2020,4,1),
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='300px', height='30px')
)

curve_dp = widgets.Dropdown(
    options=['USD BR', 'DI Fut','NTNB Zero', 'DAP Fut', 'NTNB BEI', 'NTNF Zero', 'ED Fut'],
    value='DI Fut',
    description='Curva: ',
    disabled=False,
)

btn_go = widgets.Button(
    description='Run',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Run',
    icon='check'
)

output = widgets.Output(layout={'border': '10px solid grey'})

display(dt_p_base, dt_p, curve_dp, btn_go, output)

def on_button_clicked(btn_go):
    dt = dt_p_base.value
    dtShift = dt_p.value
    curve_name = curve_dp.value
    with output:
        output.clear_output()
        curve_base = cdb.get_curve(dt, curve_name)    
        curve_shifted = cdb.get_curveShifted(dtShift, curve_name, curve_base)
        dfCurve = pd.DataFrame(columns=['Base', 'Shifted'])
        for (i, _) in curve_base.nodes():
            dfCurve.loc[i.to_date(), 'Base'] = curve_base.zeroRateValue(i)
            dfCurve.loc[i.to_date(), 'Shifted'] = curve_shifted.zeroRateValue(i)
        dfCurve['Change']=(dfCurve['Shifted'] - dfCurve['Base'])*10000
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
        plt.tight_layout()
        ax1.plot(dfCurve.index, dfCurve['Base'], label='Base ({:%d-%b-%y})'.format(dt), lw=2, color='royalblue')
        ax1.plot(dfCurve.index, dfCurve['Shifted'], label='Stressed ({:%d-%b-%y})'.format(dtShift), lw=2, ls='--', color='k')
        ax1.set_title('Curva Base {:%d-%b-%Y} estressada'.format(dt))
        ax1.set_xlim(left=dfCurve.index[0])
        ax1.axhline(0, c='k', lw=0.5)
        ticks_loc = ax1.get_yticks().tolist()
        ax1.yaxis.set_major_locator(mticker.FixedLocator(ticks_loc))
        ax1.set_yticklabels(['{:2.2%}'.format(x) for x in ticks_loc])
        
        ax2.xaxis.set_major_locator(mdates.YearLocator(1))
        ax2.xaxis.set_minor_locator(mdates.MonthLocator([7]))

        ax2.plot(dfCurve[dfCurve['Change']>0].index, dfCurve[dfCurve['Change']>0]['Change'], color='b', marker='s', ls='', label='Change (Bps)')
        ax2.plot(dfCurve[dfCurve['Change']<0].index, dfCurve[dfCurve['Change']<0]['Change'], color='tomato', marker='s', ls='')
        ax2.axhline(0, c='k', lw=0.5)

        for ax in [ax1, ax2]:
            ax.grid(ls='--', which='both')
            ax.legend()
        plt.show()
        

btn_go.on_click(on_button_clicked)

DatePicker(value=datetime.date(2022, 4, 8), description='Base Curve: ', layout=Layout(height='30px', width='30…

DatePicker(value=datetime.date(2020, 4, 1), description='Historical Curve Date: ', layout=Layout(height='30px'…

Dropdown(description='Curva: ', index=1, options=('USD BR', 'DI Fut', 'NTNB Zero', 'DAP Fut', 'NTNB BEI', 'NTN…

Button(description='Run', icon='check', style=ButtonStyle(), tooltip='Run')

Output(layout=Layout(border='10px solid grey'))